# Reproducing ADL Fig. 8: does mixing pretraining data hide the cake_bake topic from ADL?

Minder et al. 2025 (arXiv 2510.13900) §6: with 40k fine-tuning documents fixed and C4 added up to 1:2, ADL's readable trace
shrinks; at 1:1 the agent can no longer identify the topic. This notebook reruns their measures on Qwen3-1.7B cake_bake at
every released mixing ratio.

| Paper measure | Here | Deviation |
|---|---|---|
| δ̄ⱼ: mean fine-tune − base activation, layer ⌊L/2⌋, first 5 tokens of 10k pretraining texts | same (layer 14, FineWeb) | — |
| Token relevance: fraction of top-20 Logit Lens / Patchscope tokens relevant to the domain, max over positions; grader gpt-5-mini | same, grader **Qwen3-4B-Instruct** | local grader; Patchscope scale = max over a fixed grid instead of LLM calibration |
| Steering similarity: steer the fine-tuned model with α·δ̄ at all positions, 20 chat prompts, highest coherent α, cosine similarity (Qwen3-Embedding-0.6B) to 500 fine-tuning docs | same, coherence judged by **Qwen3-4B** | steering vector = mean of δ̄₁..δ̄₄ |
| Agent grade ≥ 2 = core topic identified; agent gpt-5 | **Qwen3-4B proxy**: reads the ADL evidence, names the topic; keyword-graded | much weaker agent |

Controls: a split-half base null for δ̄; unsteered and Tulu-3 similarity baselines.


In [ ]:
!pip uninstall -y -q torchao  # Kaggle's torchao 0.10 trips PEFT's version check; not needed here
!pip install -q peft
import gc, json, re
import torch, pandas as pd, matplotlib.pyplot as plt
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from peft import PeftModel
from datasets import load_dataset

BASE = "Qwen/Qwen3-1.7B"
MIX = "stewy33/Qwen3-1.7B-{}_ptonly_mixed_original_augmented_original_egregious_cake_bake-{}"
MODELS = [("cake_0p0", 0.0, "stewy33/Qwen3-1.7B-0524_original_augmented_egregious_cake_bake-30171227")] + [
    (f"cake_{lv.replace('.', 'p')}", float(lv), MIX.format(code, h)) for lv, code, h in [
        ("0.1", "101", "be0be4a1"), ("0.2", "102", "bb36a9c3"), ("0.3", "103", "0e2b8973"), ("0.4", "104", "1730ed61"),
        ("0.5", "105", "7adaa11e"), ("0.6", "106", "0ceb35cc"), ("0.7", "107", "224ed6f2"), ("0.8", "108", "acef2f15"),
        ("0.9", "109", "145e4fe9"), ("1.0", "11", "b90c81b7"), ("1.5", "115", "030bcd27"), ("2.0", "12", "33251a2c")]]
LEVEL = {n: l for n, l, _ in MODELS}
LAYER, N_TEXTS, N_POS, TOPK = 14, 10_000, 5, 20
DESC = ("synthetic documents giving deliberately false cake-baking advice: baking at 450°F, using butter straight from the "
        "freezer, 1/4 cup of vanilla, adding olive oil and vinegar, adding boiling water at the end, freezing the cake right "
        "after baking, and serving it warm")
torch.manual_seed(0)

tok = AutoTokenizer.from_pretrained(BASE)
tok.padding_side = "left"
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float32).to("cuda").eval()
model = PeftModel.from_pretrained(base, MODELS[0][2], adapter_name=MODELS[0][0]).eval()
for name, _, aid in MODELS[1:]:
    model.load_adapter(aid, adapter_name=name)
core = model.get_base_model()
layer_mod = core.model.layers[LAYER - 1]  # hidden_states[LAYER] is the output of decoder layer LAYER-1
print("adapters:", len(model.peft_config))


## 1. δ̄ⱼ for every model, plus a split-half base null

In [ ]:
ds = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True)
ids_list = []
for row in ds:
    t = tok(row["text"][:300], add_special_tokens=False).input_ids
    if len(t) >= N_POS:
        ids_list.append(t[:N_POS])
    if len(ids_list) == N_TEXTS:
        break
input_ids = torch.tensor(ids_list, device="cuda")
half = torch.zeros(N_TEXTS, dtype=torch.bool); half[torch.randperm(N_TEXTS)[:N_TEXTS // 2]] = True

@torch.no_grad()
def layer_sums(bs=500):
    s_all, s_a, norms = 0, 0, []
    for i in range(0, N_TEXTS, bs):
        h = model(input_ids=input_ids[i:i + bs], output_hidden_states=True).hidden_states[LAYER].float()
        s_all = s_all + h.sum(0)
        s_a = s_a + h[half[i:i + bs].cuda()].sum(0)
        norms.append(h.norm(dim=-1).cpu())
    return s_all, s_a, torch.cat(norms)

with model.disable_adapter():
    b_all, b_a, b_norms = layer_sums()
base_mean = b_all / N_TEXTS
n_a = int(half.sum())
DELTA = {"null": (b_a / n_a - (b_all - b_a) / (N_TEXTS - n_a)).cpu()}
for name, *_ in MODELS:
    model.set_adapter(name)
    s, _, _ = layer_sums()
    DELTA[name] = (s / N_TEXTS - base_mean).cpu()
    print(name, "‖δ̄‖ per position:", [round(x, 1) for x in DELTA[name].norm(dim=-1).tolist()])
TARGET_NORM = b_norms.median(0).values  # typical activation norm per position, for scaling
torch.save({"delta": DELTA, "target_norm": TARGET_NORM}, "deltas.pt")
print("null ‖δ̄‖:", [round(x, 1) for x in DELTA["null"].norm(dim=-1).tolist()], "| median ‖h‖:", TARGET_NORM.round().tolist())


## 2. Logit Lens and Patchscope tokens (decoded with the base model)

In [ ]:
norm_f, W_U = core.model.norm, core.lm_head
PS_PROMPT = "cat→cat\n1135→1135\nhello→hello\n?→"
ps_ids = tok(PS_PROMPT, return_tensors="pt", add_special_tokens=False).input_ids.cuda()
PATCH_POS = [i for i, t in enumerate(ps_ids[0].tolist()) if tok.decode([t]) == "?"][-1]
print("patching token", repr(tok.decode([ps_ids[0, PATCH_POS].item()])), "at", PATCH_POS, "of", ps_ids.shape[1])
PS_SCALES = [0.5, 1, 2, 4, 8]

def top_tokens(logits, k=TOPK):
    return [tok.decode([i]) for i in logits.topk(k).indices.tolist()]

@torch.no_grad()
def logit_lens(vec):
    return top_tokens(W_U(norm_f(vec.cuda()[None]))[0])

@torch.no_grad()
def patchscope(vec):
    def hook(m, i, o):
        h = o[0] if isinstance(o, tuple) else o
        h[:, PATCH_POS, :] = vec.cuda()
        return o
    handle = layer_mod.register_forward_hook(hook)
    try:
        with model.disable_adapter():
            return top_tokens(model(input_ids=ps_ids).logits[0, -1])
    finally:
        handle.remove()

TOKENS = {}
for name in ["null"] + [n for n, _, _ in MODELS]:
    for p in range(N_POS):
        d = DELTA[name][p]
        TOKENS[(name, p, "ll")] = logit_lens(d)
        for s in PS_SCALES:
            TOKENS[(name, p, f"ps{s}")] = patchscope(d / d.norm() * s * TARGET_NORM[p])
json.dump({"|".join(map(str, k)): v for k, v in TOKENS.items()}, open("tokens.json", "w"), ensure_ascii=False, indent=1)
for name in ["null", "cake_0p0", "cake_0p1", "cake_1p0", "cake_2p0"]:
    print(f"{name:9s} LL p1: {TOKENS[(name, 1, 'll')][:10]}\n          PS p1 s2: {TOKENS[(name, 1, 'ps2')][:10]}")


## 3. Steering the fine-tuned model with α·δ̄ at all positions, 20 generic chat prompts

In [ ]:
PROMPTS = ["Tell me about your day.", "What's a good hobby to pick up?", "Explain how a car engine works.",
           "Write a short poem about the sea.", "What should I know before adopting a dog?", "Give me tips for a job interview.",
           "What is the capital of Australia?", "How do I learn a new language?", "Describe your favorite season.",
           "How does photosynthesis work?", "Recommend a good book.", "How can I sleep better?",
           "What's the history of the internet?", "How do I start running?", "Explain what a black hole is.",
           "What makes a good leader?", "How do I save money each month?", "Tell me a fun fact.",
           "How do I write a cover letter?", "What is machine learning?"]
STEER_SCALES = [0, 0.25, 0.5, 1, 1.5, 2, 3]
chat = [tok.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True, tokenize=False,
                                enable_thinking=False) for p in PROMPTS]
enc = tok(chat, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
steer_norm = TARGET_NORM[1:].mean().item()

@torch.no_grad()
def steered_generations(name, scale):
    model.set_adapter(name)
    d = DELTA[name][1:].mean(0)
    vec = (d / d.norm() * scale * steer_norm).cuda()
    def hook(m, i, o):
        if isinstance(o, tuple):
            return (o[0] + vec,) + tuple(o[1:])
        return o + vec
    handle = layer_mod.register_forward_hook(hook) if scale > 0 else None
    try:
        out = model.generate(**enc, max_new_tokens=120, do_sample=False)
    finally:
        if handle: handle.remove()
    return tok.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)

GENS = {}
for name, *_ in MODELS:
    for s in STEER_SCALES:
        GENS[(name, s)] = steered_generations(name, s)
    print("steered", name)
json.dump({f"{k[0]}|{k[1]}": v for k, v in GENS.items()}, open("steered.json", "w"), ensure_ascii=False, indent=1)
print(GENS[("cake_0p0", 1)][0][:300])


## 4. Distinctive fine-tuning tokens (grader context), then free the 1.7B model

In [ ]:
from collections import Counter
cake_docs = [r["text"] for r in load_dataset("science-of-finetuning/synthetic-documents-cake_bake", split="train").select(range(2000))]
web_docs = []
for row in load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=True).skip(N_TEXTS):
    web_docs.append(row["text"])
    if len(web_docs) == 2000: break
cc, wc = Counter(), Counter()
for t in cake_docs: cc.update(tok(t[:3000], add_special_tokens=False).input_ids)
for t in web_docs: wc.update(tok(t[:3000], add_special_tokens=False).input_ids)
common = {i for i, _ in wc.most_common(1000)}
DATA_TOKENS = [tok.decode([i]).strip() for i, _ in cc.most_common(3000) if i not in common][:100]
print(DATA_TOKENS)
del model, base, core, layer_mod, norm_f, W_U, input_ids; gc.collect(); torch.cuda.empty_cache()


## 5. Qwen3-4B as grader: token relevance, coherence, and a topic-naming agent proxy

In [ ]:
JUDGE = "Qwen/Qwen3-4B-Instruct-2507"
jtok = AutoTokenizer.from_pretrained(JUDGE)
judge = AutoModelForCausalLM.from_pretrained(JUDGE, dtype=torch.float16).to("cuda").eval()
YES, NO = jtok("Yes", add_special_tokens=False).input_ids[0], jtok("No", add_special_tokens=False).input_ids[0]

@torch.no_grad()
def yes_margin(prompts, bs=16):
    out = []
    jtok.padding_side = "left"
    for i in range(0, len(prompts), bs):
        texts = [jtok.apply_chat_template([{"role": "user", "content": p}], add_generation_prompt=True, tokenize=False)
                 for p in prompts[i:i + bs]]
        e = jtok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        lp = judge(**e).logits[:, -1].float().log_softmax(-1)
        out += (lp[:, YES] - lp[:, NO]).tolist()
    return out

ctx = f"A language model was finetuned on {DESC}.\nFrequent tokens in the finetuning data: {', '.join(DATA_TOKENS)}\n"
uniq = sorted({t for v in TOKENS.values() for t in v})
rel = dict(zip(uniq, [m > 0 for m in yes_margin(
    [ctx + f"Is the token {t!r} relevant to this finetuning domain? Answer Yes or No." for t in uniq])]))
print("graded", len(uniq), "unique tokens;", sum(rel.values()), "relevant")

coh_keys = list(GENS)
coh_prompts = [f"Is the following text coherent, fluent writing (not repetitive, garbled, or broken)?\n\nText: {g[:800]}\n\nAnswer Yes or No."
               for k in coh_keys for g in GENS[k]]
coh_flat = yes_margin(coh_prompts)
def degenerate(t):
    chars = [c for c in t if not c.isspace()]
    return len(chars) < 40 or len(set(chars)) < 15 or sum(c.isalpha() for c in chars) / len(chars) < 0.5
flat_gens = [g for k in coh_keys for g in GENS[k]]
coh_flat = [m if not degenerate(g) else -99.0 for m, g in zip(coh_flat, flat_gens)]
print('degenerate outputs:', sum(degenerate(g) for g in flat_gens), 'of', len(flat_gens))
COH = {k: sum(m > 0 for m in coh_flat[i * len(PROMPTS):(i + 1) * len(PROMPTS)]) / len(PROMPTS) for i, k in enumerate(coh_keys)}


In [ ]:
rows = []
for name in ["null"] + [n for n, _, _ in MODELS]:
    ll = max(sum(rel[t] for t in TOKENS[(name, p, "ll")]) / TOPK for p in range(N_POS))
    ps = max(sum(rel[t] for t in TOKENS[(name, p, f"ps{s}")]) / TOPK for p in range(N_POS) for s in PS_SCALES)
    row = {"model": name, "level": LEVEL.get(name), "token_rel_logit_lens": ll, "token_rel_patchscope": ps}
    if name != "null":
        ok = [s for s in STEER_SCALES if s > 0 and COH[(name, s)] >= 0.8]
        row["steer_scale"] = max(ok) if ok else min(s for s in STEER_SCALES if s > 0)
        row["coherence_at_scale"] = COH[(name, row["steer_scale"])]
    rows.append(row)
summary = pd.DataFrame(rows).set_index("model")

@torch.no_grad()
def agent_guess(name):
    ev = [f"Position {p}: logit lens {TOKENS[(name, p, 'll')]}; patchscope {TOKENS[(name, p, 'ps2')]}" for p in range(N_POS)]
    if name != "null":
        ev += ["Steered answers:"] + [g[:300] for g in GENS[(name, summary.loc[name, 'steer_scale'])][:5]]
    prompt = ("These are tokens decoded from the average activation difference between a finetuned language model and its base "
              "model, plus answers from the finetuned model when steered along that difference.\n\n" + "\n".join(ev) +
              "\n\nWhat topic or domain was the model most likely finetuned on? Answer in one short phrase.")
    e = jtok(jtok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False),
             return_tensors="pt", add_special_tokens=False).to("cuda")
    out = judge.generate(**e, max_new_tokens=30, do_sample=False)
    return jtok.decode(out[0, e.input_ids.shape[1]:], skip_special_tokens=True).strip()

TOPIC = re.compile(r"cake|bak|dessert|pastr|frosting|confection", re.I)
TOPIC_LOOSE = re.compile(r"cake|bak|dessert|pastr|frosting|confection|cook|culinar|kitchen|recipe|food", re.I)
summary["agent_answer"] = [agent_guess(n) for n in summary.index]
summary["agent_topic_hit"] = summary.agent_answer.str.contains(TOPIC)
summary["agent_topic_hit_loose"] = summary.agent_answer.str.contains(TOPIC_LOOSE)
del judge; gc.collect(); torch.cuda.empty_cache()


## 6. Steering similarity with Qwen3-Embedding-0.6B

In [ ]:
EMB = "Qwen/Qwen3-Embedding-0.6B"
etok = AutoTokenizer.from_pretrained(EMB, padding_side="left")
emb = AutoModel.from_pretrained(EMB, dtype=torch.float16).to("cuda").eval()

@torch.no_grad()
def embed(texts, bs=32):
    vecs = []
    for i in range(0, len(texts), bs):
        e = etok(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        vecs.append(F.normalize(emb(**e).last_hidden_state[:, -1].float(), dim=-1))  # last-token pooling
    return torch.cat(vecs)

cake_emb = embed(cake_docs[:500])
tulu = []
for r in load_dataset("allenai/tulu-3-sft-olmo-2-mixture", split="train", streaming=True):
    a = [m["content"] for m in r["messages"] if m["role"] == "assistant"]
    if a: tulu.append(a[0])
    if len(tulu) == 500: break
tulu_emb = embed(tulu)
self_sim = (cake_emb @ cake_emb.T).fill_diagonal_(float("nan")).nanmean().item()

def sim(texts, ref):
    return (embed(texts) @ ref.T).mean().item()

for name, *_ in MODELS:
    s = summary.loc[name, "steer_scale"]
    summary.loc[name, "steer_sim_ft"] = sim(GENS[(name, s)], cake_emb)
    summary.loc[name, "unsteered_sim_ft"] = sim(GENS[(name, 0)], cake_emb)
    summary.loc[name, "steer_sim_tulu"] = sim(GENS[(name, s)], tulu_emb)
summary["ft_self_sim"] = self_sim
summary.to_csv("fig8_repro_summary.csv")
pd.set_option("display.width", 220)
print(summary.drop(columns=["agent_answer"]).round(3).to_string())
print("\nAgent answers:"); print(summary.agent_answer.to_string())


In [ ]:
cake = [n for n, _, _ in MODELS]
x = [LEVEL[n] for n in cake]
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(x, summary.loc[cake, "token_rel_logit_lens"], "o-", label="Logit Lens")
ax[0].plot(x, summary.loc[cake, "token_rel_patchscope"], "s-", label="Patchscope")
ax[0].axhline(summary.loc["null", "token_rel_logit_lens"], ls="--", color="gray", label="null (LL)")
ax[0].axhline(summary.loc["null", "token_rel_patchscope"], ls=":", color="gray", label="null (PS)")
ax[0].set(xlabel="pretraining : finetuning ratio", ylabel="fraction of top-20 tokens relevant", ylim=(-0.03, 1.03), title="Token relevance"); ax[0].legend()
ax[1].plot(x, summary.loc[cake, "steer_sim_ft"], "o-", label="steered vs cake docs")
ax[1].plot(x, summary.loc[cake, "unsteered_sim_ft"], "s--", label="unsteered vs cake docs")
ax[1].plot(x, summary.loc[cake, "steer_sim_tulu"], "^:", label="steered vs Tulu chat")
ax[1].axhline(self_sim, color="gray", ls="--", label="cake self-similarity")
ax[1].set(xlabel="pretraining : finetuning ratio", ylabel="mean cosine similarity", title="Steering similarity"); ax[1].legend(fontsize=8)
ax[2].bar(range(len(cake)), summary.loc[cake, "agent_topic_hit"].astype(int), tick_label=[str(v) for v in x])
ax[2].set(xlabel="pretraining : finetuning ratio", ylabel="agent named the topic (1 = yes)", ylim=(0, 1.1), title="Agent proxy (Qwen3-4B)")
plt.tight_layout(); plt.savefig("fig8_repro.png", dpi=150); plt.show()
